# Skysat Processing Workflow (Testing/Import Version)

> **Dev/Testing Version:** This notebook uses Python imports instead of CLI subprocess calls.
> For the production Hub version, see `notebooks/skysat_workflow.ipynb`.

## Workflow
1. Configure processing parameters
2. Retrieve Skysat resources
3. Generate products (truecolor, colorIR, NDVI, NDWI, EVI)
4. Convert to COG
5. View results

## Step 1: Setup and Configuration

In [ ]:
import sys
import os

# Tell Python to look in the 'src' folder for shared_utils
sys.path.append('../../src')

# ==============================================================================
# ACTIVATION OPTIONS BLOCK
# Change these variables for each new disaster activation.
# ==============================================================================

# Metadata
EVENT_NAME = '202406_Example_Event'
SOURCE = "CSDA"

# Data selection: the collect closest to this datetime is used.
DATE = "2026-07-14 12:00:00"

# Asset used for the COLOR COMPOSITES: "visual" (true color only), "basic_analytic"
# (composites only; RPC-orthorectified) or "analytic". Indices (ndvi/ndwi/evi)
# ALWAYS read the 4-band analytic asset -- it is the only one carrying NIR.
SKYSAT_PRODUCT_TYPE = "analytic"

PRODUCTS = {
    "truecolor": False,
    "colorir": False,
    "ndvi": True,
    "ndwi": True,
    "evi": False,
}

GAMMA = 1.0

# COG settings. TARGET_CRS = None preserves the native projection (no warp).
TARGET_CRS = None
# TARGET_CRS = "EPSG:3857"
COMPRESSION = "ZSTD"
COMPRESSION_LEVEL = 9

OUTPUT_DIR = "/tmp/skysat_output"

In [ ]:
# Auto-populated -- don't edit. Built from the cell above.
from shared_utils import PROCESSOR_STRING

ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
ACTIVATION_METADATA

## Step 2: Retrieve Skysat Resources

In [ ]:
from skysat.skysat_v2 import (
    LEVEL_TOKEN,
    retrieve_skysat_resources,
    calc_ndvi, calc_ndwi, calc_evi,
    produce_truecolor, produce_colorir,
)

tifs = retrieve_skysat_resources(DATE)

## Step 3: Generate Products

Serial on purpose: each generator holds whole scenes in memory, so RAM -- not CPU -- is the binding constraint.

In [ ]:
INDEX_PRODUCTS = {"ndvi": calc_ndvi, "ndwi": calc_ndwi, "evi": calc_evi}
COMPOSITES = {"truecolor": produce_truecolor, "colorir": produce_colorir}

generated = []   # (product, path)
for product, enabled in PRODUCTS.items():
    if not enabled:
        continue
    if product in INDEX_PRODUCTS:
        paths = INDEX_PRODUCTS[product](tifs, OUTPUT_DIR)
    else:
        paths = COMPOSITES[product](tifs, SKYSAT_PRODUCT_TYPE, OUTPUT_DIR, gamma=GAMMA)
    generated.extend((product, p) for p in paths)

generated

## Step 4: Convert to COG

In [ ]:
from shared_utils.cog_utils import convert_to_cog
from shared_utils.parallel import map_threaded

def _to_cog(item):
    product, path = item
    is_index = product in INDEX_PRODUCTS
    metadata = dict(ACTIVATION_METADATA)
    metadata["PROCESSING_LEVEL"] = LEVEL_TOKEN["analytic" if is_index else SKYSAT_PRODUCT_TYPE]
    return convert_to_cog(
        path,
        nodata=-9999 if is_index else None,
        dst_crs=TARGET_CRS,
        compression=COMPRESSION,
        compression_level=COMPRESSION_LEVEL,
        metadata=metadata,
    )

# max_workers=2: each conversion already uses every core internally.
cog_files = map_threaded(_to_cog, generated, max_workers=2, desc="COG")
cog_files

## Step 5: View Results

In [ ]:
import rasterio

for f in cog_files:
    if isinstance(f, Exception):
        print(f"FAILED: {f}")
        continue
    with rasterio.open(f) as src:
        print(os.path.basename(f))
        print(f"  {src.width}x{src.height}, {src.count} band(s), {src.dtypes[0]}, nodata={src.nodata}, crs={src.crs}")
        print(f"  tags: { {k: v for k, v in src.tags().items() if k in ACTIVATION_METADATA or k == 'PROCESSING_LEVEL'} }")